In [16]:
import random
from datetime import datetime
accounts = {}


# ---------------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------------
def now():
    
    return datetime.now().strftime("%d-%m-%Y %H:%M:%S")


def generate_account_number():
    
    while True:
        number = str(random.randint(1000000000, 9999999999))
        if number not in accounts:
            return number


def add_transaction(account, kind, amount, note=""):
   
    account["history"].append({
        "time": now(),
        "type": kind,
        "amount": amount,
        "note": note,
        "balance": account["balance"],
    })


def read_amount(prompt):
    
    text = input(prompt).strip()
    try:
        amount = float(text)
    except ValueError:
        print("Invalid amount. Please enter a number.")
        return None
    if amount <= 0:
        print("Amount must be greater than zero.")
        return None
    return round(amount, 2)


def read_pin(prompt):
    
    pin = input(prompt).strip()
    if len(pin) == 4 and pin.isdigit():
        return pin
    print("PIN must be exactly 4 digits.")
    return None


# ---------------------------------------------------------------------------
# Main menu operations
# ---------------------------------------------------------------------------
def create_account():
    print("\n--- CREATE ACCOUNT ---")

    name = input("Enter your name: ").strip()
    if not name or not name.replace(" ", "").isalpha():
        print("Invalid name. Use letters and spaces only.")
        return

    phone = input("Enter phone number (10 digits): ").strip()
    if len(phone) != 10 or not phone.isdigit():
        print("Invalid phone number. It must have exactly 10 digits.")
        return

    pin = read_pin("Create a 4-digit PIN: ")
    if pin is None:
        return
    if input("Confirm PIN: ").strip() != pin:
        print("PINs do not match. Account not created.")
        return

    acc_no = generate_account_number()
    accounts[acc_no] = {
        "name": name.title(),
        "phone": phone,
        "pin": pin,
        "balance": 0.0,
        "history": [],
    }
    print("\nAccount created successfully!")
    print(f"Account Holder : {name.title()}")
    print(f"Account Number : {acc_no}  (please remember this)")


def login():
    print("\n--- LOGIN ---")
    acc_no = input("Enter account number: ").strip()

    if acc_no not in accounts:
        print("Account not found.")
        return None

    # Allow 3 attempts for the PIN
    for attempt in range(3):
        pin = input("Enter PIN: ").strip()
        if pin == accounts[acc_no]["pin"]:
            print(f"\nLogin successful. Welcome, {accounts[acc_no]['name']}!")
            return acc_no
        print(f"Incorrect PIN. Attempts left: {2 - attempt}")

    print("Too many failed attempts. Returning to main menu.")
    return None


# ---------------------------------------------------------------------------
# Account menu operations
# ---------------------------------------------------------------------------
def check_balance(account):
    print(f"\nCurrent balance: Rs. {account['balance']:.2f}")


def deposit(account):
    amount = read_amount("Enter amount to deposit: Rs. ")
    if amount is None:
        return
    account["balance"] += amount
    add_transaction(account, "Deposit", amount)
    print(f"Rs. {amount:.2f} deposited. New balance: Rs. {account['balance']:.2f}")


def withdraw(account):
    amount = read_amount("Enter amount to withdraw: Rs. ")
    if amount is None:
        return
    if amount > account["balance"]:
        print("Insufficient balance.")
        return
    account["balance"] -= amount
    add_transaction(account, "Withdrawal", amount)
    print(f"Rs. {amount:.2f} withdrawn. New balance: Rs. {account['balance']:.2f}")


def transfer(acc_no, account):
    receiver_no = input("Enter receiver account number: ").strip()

    if receiver_no == acc_no:
        print("You cannot transfer money to your own account.")
        return
    if receiver_no not in accounts:
        print("Receiver account not found.")
        return

    receiver = accounts[receiver_no]
    amount = read_amount("Enter amount to transfer: Rs. ")
    if amount is None:
        return
    if amount > account["balance"]:
        print("Insufficient balance.")
        return

    account["balance"] -= amount
    receiver["balance"] += amount
    add_transaction(account, "Transfer Sent", amount, f"To {receiver_no} ({receiver['name']})")
    add_transaction(receiver, "Transfer Received", amount, f"From {acc_no} ({account['name']})")
    print(f"Rs. {amount:.2f} transferred to {receiver['name']}.")
    print(f"New balance: Rs. {account['balance']:.2f}")


def transaction_history(account):
    print("\n--- TRANSACTION HISTORY ---")
    if not account["history"]:
        print("No transactions yet.")
        return

    for i, t in enumerate(account["history"], start=1):
        line = f"{i}. [{t['time']}] {t['type']:<18} Rs. {t['amount']:>10.2f}  Balance: Rs. {t['balance']:.2f}"
        if t["note"]:
            line += f"  ({t['note']})"
        print(line)


def change_pin(account):
    if input("Enter old PIN: ").strip() != account["pin"]:
        print("Incorrect old PIN.")
        return

    new_pin = read_pin("Enter new PIN (4 digits): ")
    if new_pin is None:
        return
    if new_pin == account["pin"]:
        print("New PIN must be different from the old PIN.")
        return
    if input("Confirm new PIN: ").strip() != new_pin:
        print("PINs do not match. PIN not changed.")
        return

    account["pin"] = new_pin
    print("PIN changed successfully.")


def account_menu(acc_no):
    
    account = accounts[acc_no]

    while True:
        print("\n===== ACCOUNT MENU =====")
        print("1. Check Balance")
        print("2. Deposit")
        print("3. Withdraw")
        print("4. Transfer")
        print("5. Transaction History")
        print("6. Change PIN")
        print("7. Logout")
        choice = input("Enter your choice (1-7): ").strip()

        if choice == "1":
            check_balance(account)
        elif choice == "2":
            deposit(account)
        elif choice == "3":
            withdraw(account)
        elif choice == "4":
            transfer(acc_no, account)
        elif choice == "5":
            transaction_history(account)
        elif choice == "6":
            change_pin(account)
        elif choice == "7":
            print("Logged out successfully.")
            break
        else:
            print("Invalid choice. Please enter a number from 1 to 7.")


# ---------------------------------------------------------------------------
# Program entry point
# ---------------------------------------------------------------------------
def main():
    while True:
        print("\n===== WELCOME TO PYTHON BANK =====")
        print("1. Create Account")
        print("2. Login")
        print("3. Exit")
        choice = input("Enter your choice (1-3): ").strip()

        if choice == "1":
            create_account()
        elif choice == "2":
            acc_no = login()
            if acc_no:
                account_menu(acc_no)
        elif choice == "3":
            print("Thank you for banking with us.")
            break
        else:
            print("Invalid choice. Please enter 1, 2 or 3.")


if __name__ == "__main__":
    main()


===== WELCOME TO PYTHON BANK =====
1. Create Account
2. Login
3. Exit


Enter your choice (1-3):  1



--- CREATE ACCOUNT ---


Enter your name:  Kinjal
Enter phone number (10 digits):  9878765654
Create a 4-digit PIN:  3450
Confirm PIN:  3450



Account created successfully!
Account Holder : Kinjal
Account Number : 1807626055  (please remember this)

===== WELCOME TO PYTHON BANK =====
1. Create Account
2. Login
3. Exit


...


--- TRANSACTION HISTORY ---
1. [21-09-2026 22:34:08] Deposit            Rs.   25000.00  Balance: Rs. 25000.00
2. [21-09-2026 22:34:13] Withdrawal         Rs.    5000.00  Balance: Rs. 20000.00

===== ACCOUNT MENU =====
1. Check Balance
2. Deposit
3. Withdraw
4. Transfer
5. Transaction History
6. Change PIN
7. Logout


Enter your choice (1-7):  7


Logged out successfully.

===== WELCOME TO PYTHON BANK =====
1. Create Account
2. Login
3. Exit


Enter your choice (1-3):  3


Thank you for banking with us.
